# 📓 Notebook 2 — Simulación Numérica del Sistema Ecológico-Urbano

**Sección 2 — Modelación y Gamificación** · Día 3 (mañana)

### Objetivo de hoy
Traducir el modelo mental (su diagrama causal formalizado ayer) a una **simulación numérica funcional en Python**.

Al final de esta sesión tendrán un script que simula, paso a paso (año por año, o turno por turno), cómo evolucionan las variables de su sistema ecológico-urbano.

No se preocupen si nunca han programado: vamos a construir todo **línea por línea**, explicando cada parte.


## 0. Conectemos con lo que ya saben

Ayer formalizaron su modelo mental: variables, relaciones, ciclos de retroalimentación y una fórmula "en palabras" para cada variable (Notebook 1, Sección 5).

**✏️ Antes de ver una sola línea de código de simulación, respondan:**

1. Sin correr nada todavía, ¿su equipo cree que su sistema se **estabiliza** con el tiempo (llega a un punto y ya no cambia), **colapsa** (alguna variable se va a 0 o al máximo) u **oscila** (sube y baja sin parar)? Justifiquen con base en los ciclos que encontraron ayer.
2. ¿Qué variable de su modelo esperan que cambie **más rápido**? ¿Cuál casi no se moverá?

*Guarden esta predicción — la revisaremos con datos reales en la Sección 4.*


## 1. Mini-lección: ¿qué es una simulación?

Hay dos ideas clave que usaremos hoy:

**Simulación de eventos discretos / por turnos**
En vez de modelar el tiempo de forma continua, avanzamos en "saltos" (turnos, días, años). En cada turno, calculamos el nuevo valor de cada variable a partir de los valores del turno anterior — igual que en un juego de mesa donde cada jugador tira el dado por turno.

**Modelos basados en agentes (idea básica)**
En sistemas más complejos, en vez de simular "el promedio de la ciudad", se simula a cada agente (una persona, una fábrica, un parque) individualmente, y el comportamiento global emerge de sus interacciones. Hoy haremos la versión más simple: variables agregadas (promedios de toda la zona). Si les sobra tiempo, hay un reto opcional al final para dar el salto a agentes individuales.

**¿Por qué NumPy y `random`?**
- `numpy` nos deja hacer operaciones matemáticas rápidas y trabajar con listas de números (arreglos).
- `random` agrega variabilidad — el mundo real no es perfectamente predecible, así que agregamos algo de "ruido" aleatorio a la simulación.

**🤔 Antes de continuar:** ¿por qué creen que conviene simular por turnos discretos en vez de intentar modelar "cada instante" de forma continua? Piensen en cómo también ustedes mismos analizaron el tiempo en la Sección 1 (imágenes satelitales de fechas específicas, no de "todo el tiempo").


In [ ]:
import numpy as np
import random
import json
import matplotlib.pyplot as plt

# Fijamos una "semilla" aleatoria para que todos obtengan resultados reproducibles al inicio.
# (Después pueden quitar esta línea para explorar variabilidad real.)
random.seed(42)
np.random.seed(42)

print("Librerías cargadas: numpy, random, matplotlib ✅")


## 2. Cargar el modelo del Notebook 1

Si ayer guardaron su archivo `modelo_mental.json`, súbanlo a esta misma carpeta de Colab/Jupyter y ejecuten la celda de abajo para cargarlo automáticamente. Si no lo tienen a la mano, la celda usa un modelo de ejemplo para que puedan seguir avanzando.


In [ ]:
import os

if os.path.exists("modelo_mental.json"):
    with open("modelo_mental.json", "r", encoding="utf-8") as f:
        modelo = json.load(f)
    print("✅ Modelo cargado desde modelo_mental.json")
else:
    print("⚠️ No se encontró modelo_mental.json — usando un modelo de ejemplo.")
    modelo = {
        "equipo": "Equipo de ejemplo",
        "zona_de_estudio": "Zona de ejemplo",
        "variables": [
            {"nombre": "vegetacion", "descripcion": "Cobertura vegetal", "unidad": "%", "rango": [0, 100], "valor_inicial": 60},
            {"nombre": "construccion", "descripcion": "Área construida", "unidad": "%", "rango": [0, 100], "valor_inicial": 30},
            {"nombre": "poblacion", "descripcion": "Habitantes (miles)", "unidad": "miles", "rango": [0, 500], "valor_inicial": 100},
            {"nombre": "contaminacion", "descripcion": "Nivel de contaminación", "unidad": "índice", "rango": [0, 100], "valor_inicial": 20},
            {"nombre": "agua", "descripcion": "Calidad/disponibilidad de agua", "unidad": "índice", "rango": [0, 100], "valor_inicial": 70},
        ],
        "relaciones": [
            {"origen": "construccion", "destino": "vegetacion", "signo": -1, "fuerza": 0.4, "explicacion": "Más construcción reduce vegetación"},
            {"origen": "construccion", "destino": "poblacion", "signo": 1, "fuerza": 0.3, "explicacion": "Más construcción atrae población"},
            {"origen": "poblacion", "destino": "contaminacion", "signo": 1, "fuerza": 0.3, "explicacion": "Más población, más contaminación"},
            {"origen": "vegetacion", "destino": "contaminacion", "signo": -1, "fuerza": 0.35, "explicacion": "Vegetación reduce contaminación"},
            {"origen": "contaminacion", "destino": "agua", "signo": -1, "fuerza": 0.3, "explicacion": "Contaminación reduce calidad del agua"},
            {"origen": "agua", "destino": "vegetacion", "signo": 1, "fuerza": 0.25, "explicacion": "Más agua favorece vegetación"},
        ],
    }

print(f"Equipo: {modelo['equipo']} | Zona: {modelo['zona_de_estudio']}")
for v in modelo["variables"]:
    print(f"  - {v['nombre']}: inicial={v['valor_inicial']}, rango={v['rango']}")


## 3. De variables y relaciones a código: la clase `EcosistemaUrbano`

Aquí traducimos el diagrama causal a Python. La idea central:

```
nuevo_valor = valor_actual + suma_de_efectos(relaciones que lo afectan) + un poco de ruido aleatorio
```

Cada relación `{origen, destino, signo, fuerza}` se convierte en:

```
efecto = signo * fuerza * (valor_normalizado_del_origen)
```

Y "normalizado" quiere decir: llevamos el valor a una escala de 0 a 1 dividiendo por su rango máximo, para que las fuerzas (0.0–1.0) tengan sentido sin importar si la variable va de 0-100 o de 0-500.


In [ ]:
class EcosistemaUrbano:
    """Simulación numérica simple del sistema ecológico-urbano de un equipo."""

    def __init__(self, modelo, ruido=0.03):
        self.variables_info = {v["nombre"]: v for v in modelo["variables"]}
        self.relaciones = modelo["relaciones"]
        self.ruido = ruido  # qué tanta variabilidad aleatoria agregamos (0 = nada, 0.1 = bastante)

        # estado actual: nombre -> valor
        self.estado = {v["nombre"]: float(v["valor_inicial"]) for v in modelo["variables"]}

        # guardamos el historial completo para poder graficarlo después
        self.historial = {nombre: [valor] for nombre, valor in self.estado.items()}
        self.turno = 0

    def _normalizar(self, nombre, valor):
        minimo, maximo = self.variables_info[nombre]["rango"]
        if maximo == minimo:
            return 0.0
        return (valor - minimo) / (maximo - minimo)

    def _clip(self, nombre, valor):
        minimo, maximo = self.variables_info[nombre]["rango"]
        return max(minimo, min(maximo, valor))

    def calcular_efectos(self):
        """Calcula, para cada variable, la suma de efectos que recibe de sus causas."""
        efectos = {nombre: 0.0 for nombre in self.estado}
        for r in self.relaciones:
            origen_normalizado = self._normalizar(r["origen"], self.estado[r["origen"]])
            efecto = r["signo"] * r["fuerza"] * origen_normalizado
            efectos[r["destino"]] += efecto
        return efectos

    def paso(self, acciones=None):
        """Avanza la simulación un turno. 'acciones' permite sumar efectos extra
        (por ejemplo, decisiones del jugador) — lo usaremos en el Notebook 3 y 4."""
        acciones = acciones or {}
        efectos = self.calcular_efectos()

        nuevo_estado = {}
        for nombre, valor in self.estado.items():
            minimo, maximo = self.variables_info[nombre]["rango"]
            escala = (maximo - minimo)

            cambio_por_relaciones = efectos[nombre] * escala * 0.15   # qué tan rápido "pesa" cada relación
            cambio_por_accion = acciones.get(nombre, 0.0)
            cambio_aleatorio = np.random.normal(0, self.ruido) * escala

            nuevo_valor = valor + cambio_por_relaciones + cambio_por_accion + cambio_aleatorio
            nuevo_estado[nombre] = self._clip(nombre, nuevo_valor)

        self.estado = nuevo_estado
        self.turno += 1
        for nombre, valor in self.estado.items():
            self.historial[nombre].append(valor)

        return self.estado

    def simular(self, n_turnos):
        for _ in range(n_turnos):
            self.paso()
        return self.historial


print("Clase EcosistemaUrbano definida ✅")


### 🔍 Antes de correr nada — revisen el código

**✏️ Discutan en equipo y respondan:**

- En el método `paso()`, ¿qué pasaría si borráramos la línea de `cambio_aleatorio`? ¿La simulación sería más o menos parecida a la realidad? *(reemplazar)*
- ¿Por qué creen que se usa `self._clip(...)` después de calcular cada `nuevo_valor`? ¿Qué problema evita? *(reemplazar)*


## 4. Prueba inicial: correr la simulación con valores base

Vamos a crear una instancia del ecosistema con los valores iniciales de su modelo y correrla por 30 turnos (por ejemplo, 30 años) **sin ninguna intervención del jugador**, solo para ver cómo se comporta "sola".

**🤔 Última oportunidad de predecir antes de ver el resultado real:** ¿mantienen la predicción que escribieron en la Sección 0 (estabiliza / colapsa / oscila, y qué variable cambia más)? Si cambiaron de opinión después de ver el código, anótenlo.


In [ ]:
eco = EcosistemaUrbano(modelo, ruido=0.03)
historial = eco.simular(n_turnos=30)

for nombre in historial:
    valor_inicial = historial[nombre][0]
    valor_final = historial[nombre][-1]
    tendencia = "⬆️ subió" if valor_final > valor_inicial else ("⬇️ bajó" if valor_final < valor_inicial else "➡️ igual")
    print(f"{nombre}: {valor_inicial:.1f} → {valor_final:.1f}  ({tendencia})")


In [ ]:
# Visualización: evolución de todas las variables a lo largo de los turnos

plt.figure(figsize=(9, 5))
for nombre, valores in historial.items():
    plt.plot(valores, label=nombre, linewidth=2)

plt.xlabel("Turno (ej. años)")
plt.ylabel("Valor de la variable")
plt.title(f"Simulación base — {modelo['equipo']} ({modelo['zona_de_estudio']})")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### 🔍 Observen y reflexionen

**✏️ Comparen la gráfica real con su predicción de la Sección 0. Escriban su respuesta:**

- ¿Acertaron en si el sistema se estabiliza, colapsa u oscila? Si no acertaron, ¿qué relación de su modelo explica el comportamiento real que no habían anticipado? *(reemplazar)*
- ¿Acertaron en cuál variable cambiaría más rápido? *(reemplazar)*
- Señalen un ciclo de su Notebook 1 y encuéntrenlo "en acción" en esta gráfica: ¿en qué tramo de la curva se nota su efecto? *(reemplazar)*


## 5. Experimenten: ¿qué pasa si cambiamos los parámetros?

Prueben modificar:
- El **ruido** (variabilidad aleatoria): ¿qué pasa con `ruido=0.0` vs `ruido=0.1`?
- El **número de turnos**: ¿el sistema se estabiliza, colapsa, u oscila si simulan 100 turnos en vez de 30?
- Los **valores iniciales** en `modelo_mental.json`: cambien un valor inicial y vuelvan a correr.

**🤔 Antes de tocar el código:** elijan uno de los tres experimentos de arriba y predigan qué esperan que pase. Luego corran la celda y comprueben.

Usen la celda de abajo para experimentar libremente.


In [ ]:
# ✏️ EXPERIMENTA aquí: cambia ruido y n_turnos, y vuelve a correr esta celda.

eco_experimento = EcosistemaUrbano(modelo, ruido=0.08)   # prueba 0.0, 0.03, 0.08, 0.15...
historial_experimento = eco_experimento.simular(n_turnos=60)  # prueba 10, 30, 60, 100...

plt.figure(figsize=(9, 5))
for nombre, valores in historial_experimento.items():
    plt.plot(valores, label=nombre, linewidth=2)
plt.xlabel("Turno")
plt.ylabel("Valor")
plt.title("Experimento: ¿qué cambia con más ruido / más turnos?")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### 🔍 Observen y reflexionen

**✏️ Escriban su respuesta:**

- ¿Se cumplió su predicción del experimento que eligieron? *(reemplazar)*
- Prueben `ruido=0.0` exactamente. ¿El sistema se vuelve completamente predecible? ¿Por qué eso puede ser un problema si quisiéramos representar una ciudad real? *(reemplazar)*


## 6. Guía rápida de prompts para usar IA como copiloto de código

Cuando se atoren programando, prueben pedirle ayuda a una IA (ChatGPT/Claude) con prompts como estos — **entre más específicos, mejor la respuesta**:

1. *"Tengo esta función en Python que simula una variable con un valor entre 0 y 100. Me está dando un error `KeyError: 'poblacion'`. Aquí está mi código: [pegar código]. ¿Qué está pasando?"*
2. *"Quiero agregar una nueva variable llamada 'trafico' a mi simulación en esta clase de Python [pegar clase EcosistemaUrbano]. Debe subir cuando sube 'poblacion' y bajar cuando sube 'vegetacion'. ¿Cómo la agrego sin romper el resto del código?"*
3. *"Explícame línea por línea qué hace esta función, como si tuviera 15 años y nunca hubiera programado: [pegar función `calcular_efectos`]"*
4. *"¿Cómo puedo graficar solo dos de las cinco variables de mi historial, en vez de las cinco juntas?"*

**Regla de oro:** siempre lean y entiendan el código que la IA les da antes de copiarlo — el objetivo es que ustedes entiendan su propio modelo, no que la IA lo entienda por ustedes.


## 7. Guardar el script de simulación (producto de la sesión)

Esta celda exporta la simulación a un archivo `.py` independiente — el **producto esperado de hoy**: *"Script de Python con simulación numérica básica del sistema ecológico-urbano."* Este mismo archivo lo reutilizaremos como base en los Notebooks 3 y 4.


In [ ]:
codigo_simulacion = '''
import numpy as np
import json


class EcosistemaUrbano:
    # Simulacion numerica simple del sistema ecologico-urbano de un equipo.

    def __init__(self, modelo, ruido=0.03):
        self.variables_info = {v["nombre"]: v for v in modelo["variables"]}
        self.relaciones = modelo["relaciones"]
        self.ruido = ruido
        self.estado = {v["nombre"]: float(v["valor_inicial"]) for v in modelo["variables"]}
        self.historial = {nombre: [valor] for nombre, valor in self.estado.items()}
        self.turno = 0

    def _normalizar(self, nombre, valor):
        minimo, maximo = self.variables_info[nombre]["rango"]
        if maximo == minimo:
            return 0.0
        return (valor - minimo) / (maximo - minimo)

    def _clip(self, nombre, valor):
        minimo, maximo = self.variables_info[nombre]["rango"]
        return max(minimo, min(maximo, valor))

    def calcular_efectos(self):
        efectos = {nombre: 0.0 for nombre in self.estado}
        for r in self.relaciones:
            origen_normalizado = self._normalizar(r["origen"], self.estado[r["origen"]])
            efecto = r["signo"] * r["fuerza"] * origen_normalizado
            efectos[r["destino"]] += efecto
        return efectos

    def paso(self, acciones=None):
        acciones = acciones or {}
        efectos = self.calcular_efectos()
        nuevo_estado = {}
        for nombre, valor in self.estado.items():
            minimo, maximo = self.variables_info[nombre]["rango"]
            escala = (maximo - minimo)
            cambio_por_relaciones = efectos[nombre] * escala * 0.15
            cambio_por_accion = acciones.get(nombre, 0.0)
            cambio_aleatorio = np.random.normal(0, self.ruido) * escala
            nuevo_valor = valor + cambio_por_relaciones + cambio_por_accion + cambio_aleatorio
            nuevo_estado[nombre] = self._clip(nombre, nuevo_valor)
        self.estado = nuevo_estado
        self.turno += 1
        for nombre, valor in self.estado.items():
            self.historial[nombre].append(valor)
        return self.estado

    def simular(self, n_turnos):
        for _ in range(n_turnos):
            self.paso()
        return self.historial


if __name__ == "__main__":
    with open("modelo_mental.json", "r", encoding="utf-8") as f:
        modelo = json.load(f)
    eco = EcosistemaUrbano(modelo)
    historial = eco.simular(30)
    for nombre, valores in historial.items():
        print(nombre, "->", round(valores[-1], 1))
'''

with open("simulacion_ecosistema.py", "w", encoding="utf-8") as f:
    f.write(codigo_simulacion)

print("✅ Script guardado como 'simulacion_ecosistema.py'")


## 🚀 Reto opcional (si les sobra tiempo)

Den el salto de "variables agregadas" a un **mini modelo basado en agentes**: en vez de una sola variable `poblacion`, simulen una lista de 20-30 "habitantes" (cada uno un diccionario con, por ejemplo, `satisfaccion`) y calculen `poblacion` como el promedio. Pista: usen una lista de diccionarios y un ciclo `for` para actualizar a cada agente en cada turno.


## 🧭 Bitácora de aprendizaje

**✏️ Completen cada punto como equipo:**

1. En sus propias palabras, ¿qué significa que un modelo sea "una simplificación de la realidad"? Den un ejemplo concreto de algo que su `EcosistemaUrbano` NO puede capturar de su ciudad real. *(reemplazar)*
2. ¿Qué se gana y qué se pierde al "normalizar" las variables (llevarlas todas a una escala de 0 a 1) para poder compararlas? *(reemplazar)*
3. De todo lo que programaron hoy, ¿qué parte les costó más entender? ¿Cómo la resolvieron (solos, con su equipo, con la IA, con el instructor)? *(reemplazar)*
4. ¿Qué le preguntarían a un urbanista o funcionario de su ciudad para mejorar la precisión de su simulación? *(reemplazar)*

## ✅ Producto esperado de esta sesión
- [ ] Clase `EcosistemaUrbano` funcionando con las variables y relaciones de su propio modelo
- [ ] Simulación corrida con valores base (mínimo 30 turnos) y gráfica generada
- [ ] Al menos un experimento documentado (cambiar ruido, turnos o valores iniciales)
- [ ] Archivo `simulacion_ecosistema.py` exportado
- [ ] Bitácora de aprendizaje completa

**Esta tarde:** Notebook 3 — construimos la interfaz del videojuego y conectamos esta simulación con un mapa interactivo. 🎮
